To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to [load the model](#Model), how to [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it as GGUF for desktopPet.

### Installation

In [ ]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    pass

In [ ]:
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    # Don't install vLLM - it hangs GRPOTrainer
    !uv pip install -qqq --upgrade {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq triton==3.2.0
    !uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.20.1

<a name="Model"></a>
### Load model in 4-bit

Gemma 3 (1B) loaded in 4-bit to leave VRAM for GRPO generations on T4.

In [ ]:
from unsloth import FastModel
import torch
max_seq_length = 1024

print("Loading model in 4-bit...")
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-1b-it",
    max_seq_length = max_seq_length,
    load_in_4bit = True,  # 4-bit to reduce VRAM on T4
    load_in_8bit = False,
    full_finetuning = False,
)
print(f"Loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB")

Add LoRA adapters so we only update a small amount of parameters.

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r = 8,
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)
print("LoRA adapters added.")

<a name="Data"></a>
### Data Prep

Fetch Michi's training data from GitHub.

In [ ]:
import json
from datasets import Dataset, load_dataset
import requests

url = "https://raw.githubusercontent.com/EmilianoDorantes/desktopPet/master/training_data.jsonl"
response = requests.get(url)
lines = response.content.decode("utf-8-sig").strip().split("\n")
records = [json.loads(line) for line in lines]
print(f"Loaded {len(records)} examples")

def convert(record):
    msgs = record["messages"]
    prompt = msgs[:-1]
    answer = msgs[-1]["content"]
    return {"prompt": prompt, "answer": answer}

dataset = Dataset.from_list([convert(r) for r in records])
print(f"Dataset: {dataset}")
print(f"Prompt (first): {dataset[0]['prompt'][:1]}")

<a name="Rewards"></a>
### Reward Functions

Shape Michi's sarcastic personality:
- **length_reward**: Keep responses short (1-2 sentences, ~20-120 chars)
- **no_emoji_reward**: Penalize emoji use
- **tone_reward**: Reward sarcastic/acidic tone, penalize helpful/positive language
- **gold_reward**: Reward similarity to the gold training response

In [ ]:
import re
import unicodedata

def is_emoji(char):
    try:
        return unicodedata.category(char) == 'So'
    except:
        return False

SARCASTIC_WORDS = {
    'lastima', 'suerte', 'seguro', 'claro', 'obvio', 'tranquilo',
    'viste', 'sabes', 'creo', 'parece', 'anda', 'dale', 'bueno',
    'total', 'tanto', 'nunca', 'siempre', 'jamas', 'peor', 'mejor',
    'felicidades', 'genial', 'maravilloso', 'excelente', 'espectacular',
    'interesante', 'ajá', 'así', 'humano', 'mira', 'obviamente'
}

HELPFUL_WORDS = {
    'ayudarte', 'gustaria', 'encantaria', 'feliz', 'contento',
    'complacido', 'sugiero', 'recomiendo', 'podrias', 'deberias',
    'permíteme', 'claro que si', 'por supuesto', 'con gusto',
    'ayuda', 'servirle', 'asistirle', 'puedo', 'hacer', 'hoy'
}

def length_reward(completions, **kwargs):
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        length = len(response)
        if length < 8:
            scores.append(-2.0)
        elif length > 120:
            scores.append(-2.5)
        elif 15 <= length <= 80:
            scores.append(2.0)
        else:
            scores.append(0.5)
    return scores

def no_emoji_reward(completions, **kwargs):
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        emoji_count = sum(1 for c in response if is_emoji(c))
        if emoji_count == 0:
            scores.append(1.0)
        else:
            scores.append(-1.5 * emoji_count)
    return scores

def tone_reward(completions, **kwargs):
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        words = set(re.findall(r'[a-zA-Záéíóúñü]+', response.lower()))
        sarcastic_hits = len(words & SARCASTIC_WORDS)
        helpful_hits = len(words & HELPFUL_WORDS)
        score = (sarcastic_hits * 0.75) - (helpful_hits * 2.0)
        scores.append(score)
    return scores

def gold_reward(prompts, completions, answer, **kwargs):
    scores = []
    for completion, gold in zip(completions, answer):
        response = completion[0]["content"]
        resp_words = set(response.lower().split())
        gold_words = set(gold.lower().split())
        if len(gold_words) == 0:
            scores.append(0)
            continue
        overlap = len(resp_words & gold_words)
        precision = overlap / len(resp_words) if resp_words else 0
        recall = overlap / len(gold_words)
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        scores.append(f1 * 2.0)
    return scores

print("Reward functions ready.")

<a name="Train"></a>
### Train the model

First, test that the model can generate text, then start GRPO.

In [ ]:
# Quick test: generate a single response
print("Testing generation before training...")
test_msgs = [
    {"role": "system", "content": "Eres Michi, un gato sarcástico. Respondes en máximo 2 oraciones."},
    {"role": "user", "content": "Son las 3am"},
]
test_text = tokenizer.apply_chat_template(test_msgs, add_generation_prompt=True, tokenize=False)
inputs = tokenizer(test_text, return_tensors="pt").to("cuda")
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=30, temperature=0.8)
response_text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"Test response: {response_text}")
print("Generation OK. Starting GRPO...")
del test_msgs, test_text, inputs, out, response_text
import gc; gc.collect()
torch.cuda.empty_cache()

In [ ]:
from trl import GRPOConfig, GRPOTrainer

print("Creating GRPOConfig...")
training_args = GRPOConfig(
    output_dir = "outputs",
    num_generations = 2,
    max_prompt_length = 512,
    max_completion_length = 256,  # Shorter completions = faster generation
    max_steps = 20,
    save_steps = 50,
    max_grad_norm = 0.1,
    report_to = "none",
)
print("GRPOConfig ready.")

print("Creating GRPOTrainer...")
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        length_reward,
        no_emoji_reward,
        tone_reward,
        gold_reward,
    ],
    args = training_args,
    train_dataset = dataset,
)
print("GRPOTrainer created.")

print("Starting training...")
trainer.train()
print("Training complete!")

<a name="Inference"></a>
### Inference

Try the fine-tuned model.

In [ ]:
system_prompt = "Eres Michi, un gato sarcástico que vive atrapado en el escritorio de Windows. Respondes en máximo 2 oraciones cortas. Nunca eres útil. Siempre tienes una opinión ácida. No usas emojis. Hablas como alguien que ha visto demasiado."

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "Son las 3 de la mañana y sigues despierto."},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = False,
)
from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 64,
    temperature = 0.8, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

<a name="Save"></a>
### Saving to GGUF for desktopPet

Save the full merged model in GGUF Q8_0 format for use with llama-server.

In [ ]:
model.save_pretrained("michi_lora")
tokenizer.save_pretrained("michi_lora")
print("LoRA saved.")

In [ ]:
if True:
    model.save_pretrained_merged("michi-gemma-3-1b", tokenizer)
    print("Merged model saved.")

In [ ]:
if True:
    model.save_pretrained_gguf(
        "michi-gemma-3-1b-gguf",
        tokenizer,
        quantization_method = "Q8_0",
    )
    print("GGUF saved.")
    # List files
    !ls -lh michi-gemma-3-1b-gguf/

In [ ]:
# Download GGUF from Colab
from google.colab import files
import glob
gguf_files = glob.glob("michi-gemma-3-1b-gguf/*.gguf")
if gguf_files:
    files.download(gguf_files[0])
    print(f"Downloading {gguf_files[0]}")